In [ ]:
from pathlib import Path

import cv2
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

FT_PER_METER = 3.280839895

frames_dir = "/Volumes/PortableSSD/tutrtletest/turtlepond_2026-06-29_111037/frames"
meta_csv_dir = "/Volumes/PortableSSD/tutrtletest/turtlepond_2026-06-29_111037"
frame_name = "frame_002927"
interpolation_mode = "nearest"
warp_output_scale = 5
cartesian_zoom_y_fraction = 0.8
cartesian_zoom_width_ft = 6
cartesian_zoom_height_ft = 4

img_name = f"{frame_name}_raw_rotated.png"
meta_name = f"{frame_name}_warp_metadata.txt"
theta_name = f"{frame_name}_theta.csv"

img_path = Path(frames_dir) / img_name
meta_csv_path = Path(meta_csv_dir) / meta_name
theta_path = Path(meta_csv_dir) / theta_name

In [ ]:
# show the unwarped (polar) and warped (cartesian) images side by side
def read_warp_metadata(path):
    metadata = {}
    with open(path) as f:
        for line in f:
            if ":" not in line:
                continue
            key, value = line.strip().split(":", 1)
            metadata[key.strip()] = value.strip()
    return metadata


def metadata_float(metadata, key, default=None):
    value = metadata.get(key)
    if value is None or value == "unavailable":
        return default
    return float(value)


def load_theta_degrees(path):
    theta_table = np.genfromtxt(path, delimiter=",", names=True)
    return np.asarray(theta_table["theta_degrees"], dtype=np.float32)


def load_polar_image(path, theta_count=None):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    # The rotated raw PNG for this export is range rows x theta columns.
    if theta_count is not None and image.shape[0] == theta_count and image.shape[1] != theta_count:
        image = image.T
    return image


def resolve_interpolation_mode(mode):
    interpolation_modes = {
        "nearest": cv2.INTER_NEAREST,
        "linear": cv2.INTER_LINEAR,
        "cubic": cv2.INTER_CUBIC,
        "area": cv2.INTER_AREA,
        "lanczos4": cv2.INTER_LANCZOS4,
    }
    normalized_mode = str(mode).lower()
    if normalized_mode not in interpolation_modes:
        options = ", ".join(sorted(interpolation_modes))
        raise ValueError(f"Unknown interpolation_mode {mode!r}; choose one of: {options}")
    return interpolation_modes[normalized_mode]


def center_column_valid_span(polar_image, signal_threshold=8):
    center_column = polar_image[:, polar_image.shape[1] // 2]
    signal_rows = np.flatnonzero(center_column > signal_threshold)
    if signal_rows.size == 0:
        raise ValueError("Center column has no signal above threshold; cannot infer selected range scale")
    first_valid_row = int(signal_rows[0])
    last_valid_row = int(signal_rows[-1])
    return first_valid_row, last_valid_row, last_valid_row - first_valid_row + 1


def polar_to_cartesian(
    polar_image,
    theta_degrees,
    meters_per_range_bin,
    first_valid_range_row,
    last_valid_range_row,
    selected_range_m,
    interpolation,
    output_scale=1,
):
    range_count, theta_count = polar_image.shape
    theta_degrees = np.asarray(theta_degrees, dtype=np.float32)
    if theta_degrees.size != theta_count:
        raise ValueError(
            f"Theta count ({theta_degrees.size}) does not match polar image columns ({theta_count})"
        )

    output_scale = float(output_scale)
    if output_scale <= 0:
        raise ValueError("output_scale must be greater than 0")

    max_range_m = selected_range_m
    theta_min_rad = np.deg2rad(float(np.min(theta_degrees)))
    theta_max_rad = np.deg2rad(float(np.max(theta_degrees)))
    x_min_m = max_range_m * np.sin(theta_min_rad)
    x_max_m = max_range_m * np.sin(theta_max_rad)

    output_height = int(np.ceil((last_valid_range_row - first_valid_range_row + 1) * output_scale))
    output_width = int(np.ceil(((x_max_m - x_min_m) / meters_per_range_bin) * output_scale)) + 1
    x_coords = np.linspace(x_min_m, x_max_m, output_width, dtype=np.float32)
    y_coords = np.linspace(0, max_range_m, output_height, dtype=np.float32)
    x_grid = x_coords[None, :]
    y_grid = y_coords[:, None]

    range_m_map = np.sqrt(x_grid**2 + y_grid**2)
    range_bin_map = range_m_map / meters_per_range_bin
    source_range_map = last_valid_range_row - range_bin_map
    theta_map = np.rad2deg(np.arctan2(x_grid, y_grid))
    theta_index_map = np.interp(theta_map, theta_degrees, np.arange(theta_count)).astype(np.float32)

    outside_wedge = (
        (range_bin_map < 0)
        | (range_m_map > max_range_m)
        | (source_range_map < first_valid_range_row)
        | (source_range_map > last_valid_range_row)
        | (theta_map < theta_degrees.min())
        | (theta_map > theta_degrees.max())
    )
    source_range_map = source_range_map.astype(np.float32)
    theta_index_map[outside_wedge] = -1
    source_range_map[outside_wedge] = -1

    used_polar_mask = None
    usage_stats = None
    if interpolation == cv2.INTER_NEAREST:
        valid_remap = (source_range_map >= 0) & (theta_index_map >= 0)
        used_rows = np.floor(source_range_map[valid_remap] + 0.5).astype(int)
        used_cols = np.floor(theta_index_map[valid_remap] + 0.5).astype(int)
        used_rows = np.clip(used_rows, 0, range_count - 1)
        used_cols = np.clip(used_cols, 0, theta_count - 1)

        used_polar_mask = np.zeros_like(polar_image, dtype=bool)
        used_polar_mask[used_rows, used_cols] = True

        valid_polar_mask = np.zeros_like(polar_image, dtype=bool)
        valid_polar_mask[first_valid_range_row:last_valid_range_row + 1, :] = True
        used_valid_mask = used_polar_mask & valid_polar_mask
        unused_valid_mask = valid_polar_mask & ~used_polar_mask

        valid_count = int(valid_polar_mask.sum())
        used_count = int(used_valid_mask.sum())
        unused_count = int(unused_valid_mask.sum())
        usage_stats = {
            "valid_count": valid_count,
            "used_count": used_count,
            "unused_count": unused_count,
            "unused_fraction": unused_count / valid_count if valid_count else np.nan,
        }

    cartesian = cv2.remap(
        polar_image,
        theta_index_map,
        source_range_map,
        interpolation=interpolation,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    return cartesian, (x_min_m, x_max_m, 0, max_range_m), used_polar_mask, usage_stats


def extent_m_to_ft(extent_m):
    return tuple(value * FT_PER_METER for value in extent_m)


def grid_ticks_2ft(axis_min_ft, axis_max_ft):
    start = np.floor(axis_min_ft / 2) * 2
    stop = np.ceil(axis_max_ft / 2) * 2
    return np.arange(start, stop + 0.1, 2)


def polar_pixels_to_cartesian_ft(rows, cols, theta_degrees, last_valid_range_row, feet_per_range_bin):
    ranges_ft = (last_valid_range_row - rows) * feet_per_range_bin
    theta_rad = np.deg2rad(theta_degrees[cols])
    x_ft = ranges_ft * np.sin(theta_rad)
    y_ft = ranges_ft * np.cos(theta_rad)
    return x_ft, y_ft


metadata = read_warp_metadata(meta_csv_path)
theta_degrees = load_theta_degrees(theta_path)
polar_image = load_polar_image(img_path, theta_count=len(theta_degrees))
interpolation = resolve_interpolation_mode(interpolation_mode)
selected_range_ft = metadata_float(metadata, "estimated_selected_range_feet")
selected_range_m = metadata_float(metadata, "estimated_selected_range_meters")
center_column_signal_threshold = 8
first_valid_range_row, last_valid_range_row, valid_range_rows = center_column_valid_span(
    polar_image,
    signal_threshold=center_column_signal_threshold,
)

if selected_range_m is None:
    raise ValueError("Metadata must include estimated_selected_range_meters")
meters_per_range_bin = selected_range_m / max(valid_range_rows - 1, 1)

cartesian_image, cartesian_extent, used_polar_mask, polar_usage_stats = polar_to_cartesian(
    polar_image,
    theta_degrees,
    meters_per_range_bin,
    first_valid_range_row=first_valid_range_row,
    last_valid_range_row=last_valid_range_row,
    selected_range_m=selected_range_m,
    interpolation=interpolation,
    output_scale=warp_output_scale,
)

print(f"polar image: {polar_image.shape} range x theta")
print(f"theta: {theta_degrees.min():.2f} to {theta_degrees.max():.2f} degrees")
print(
    f"center-column selected range rows: {first_valid_range_row}..{last_valid_range_row} "
    f"({valid_range_rows} rows, threshold > {center_column_signal_threshold})"
)
print(f"range scale: {meters_per_range_bin * FT_PER_METER:.4f} ft/bin")
print(f"selected range: {selected_range_ft:.2f} ft")
print(f"interpolation mode: {interpolation_mode}")
print(f"warp output scale: {warp_output_scale}x")
print(f"cartesian image: {cartesian_image.shape}")
if polar_usage_stats is not None:
    print(f"valid polar datapoints: {polar_usage_stats['valid_count']}")
    print(f"used in warped image: {polar_usage_stats['used_count']}")
    print(f"unused valid polar datapoints: {polar_usage_stats['unused_count']}")
    print(f"unused valid polar fraction: {polar_usage_stats['unused_fraction']:.2%}")
else:
    print("polar usage stats: only available for nearest interpolation")

# save the warped cartesian image to a PNG file
cartesian_output_path = f"outputs/{frame_name}_warped_cartesian.png"
cv2.imwrite(str(cartesian_output_path), cartesian_image)

above_range_rows, above_range_cols = np.nonzero(polar_image[:first_valid_range_row, :] > 0)
above_range_values = polar_image[above_range_rows, above_range_cols]
above_range_x_ft, above_range_y_ft = polar_pixels_to_cartesian_ft(
    above_range_rows,
    above_range_cols,
    theta_degrees,
    last_valid_range_row,
    meters_per_range_bin * FT_PER_METER,
)
print(f"nonzero points above selected range line: {above_range_rows.size}")

fig, ax = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)

polar_extent = (
    float(theta_degrees.min()),
    float(theta_degrees.max()),
    0,
    last_valid_range_row * meters_per_range_bin * FT_PER_METER,
)
ax[0].imshow(
    polar_image,
    cmap="gray",
    aspect="auto",
    extent=polar_extent,
    origin="upper",
    interpolation="nearest",
)
ax[0].set_title("Unwarped polar")
ax[0].set_xlabel("theta (degrees)")
ax[0].set_ylabel("range (ft)")
ax[0].axhline(selected_range_ft, color="magenta", linewidth=1.2, alpha=0.9)

cartesian_extent_ft = extent_m_to_ft(cartesian_extent)
ax[1].set_facecolor("black")
ax[1].imshow(
    cartesian_image,
    cmap="gray",
    extent=cartesian_extent_ft,
    origin="lower",
    interpolation="nearest",
)
ax[1].set_title("Warped cartesian")
ax[1].set_xlabel("x (ft)")
ax[1].set_ylabel("forward range (ft)")
ax[1].set_aspect("equal")

x_min_ft, x_max_ft, y_min_ft, y_max_ft = cartesian_extent_ft
plot_x_min_ft = min(x_min_ft, float(above_range_x_ft.min())) if above_range_x_ft.size else x_min_ft
plot_x_max_ft = max(x_max_ft, float(above_range_x_ft.max())) if above_range_x_ft.size else x_max_ft
plot_y_max_ft = max(y_max_ft, float(above_range_y_ft.max())) if above_range_y_ft.size else y_max_ft
ax[1].set_xticks(grid_ticks_2ft(plot_x_min_ft, plot_x_max_ft))
ax[1].set_yticks(np.arange(0, np.ceil(plot_y_max_ft / 2) * 2 + 0.1, 2))
ax[1].grid(color="white", alpha=0.25, linewidth=0.8)
if selected_range_ft is not None:
    range_circle = plt.Circle(
        (0, 0),
        selected_range_ft,
        fill=False,
        color="magenta",
        linewidth=1.4,
        alpha=0.9,
    )
    ax[1].add_patch(range_circle)

theta_arc = np.linspace(theta_degrees.min(), theta_degrees.max(), 400)
arc_radius_ft = (selected_range_m or cartesian_extent[3]) * FT_PER_METER
ax[1].plot(
    arc_radius_ft * np.sin(np.deg2rad(theta_arc)),
    arc_radius_ft * np.cos(np.deg2rad(theta_arc)),
    color="white",
    linestyle="--",
    linewidth=1.0,
    alpha=0.85,
)
if above_range_x_ft.size:
    ax[1].scatter(
        above_range_x_ft,
        above_range_y_ft,
        c=above_range_values,
        cmap="inferno",
        s=1,
        alpha=1.0,
        linewidths=0,
    )
ax[1].set_xlim(plot_x_min_ft, plot_x_max_ft)
ax[1].set_ylim(y_min_ft, plot_y_max_ft)

plt.show()

zoom_center_x_ft = 0
zoom_center_y_ft = y_min_ft + cartesian_zoom_y_fraction * (plot_y_max_ft - y_min_ft)
zoom_x_min_ft = max(plot_x_min_ft, zoom_center_x_ft - cartesian_zoom_width_ft / 2)
zoom_x_max_ft = min(plot_x_max_ft, zoom_center_x_ft + cartesian_zoom_width_ft / 2)
zoom_y_min_ft = max(y_min_ft, zoom_center_y_ft - cartesian_zoom_height_ft / 2)
zoom_y_max_ft = min(plot_y_max_ft, zoom_center_y_ft + cartesian_zoom_height_ft / 2)

fig_zoom, ax_zoom = plt.subplots(figsize=(8, 6), constrained_layout=True)
ax_zoom.set_facecolor("black")
ax_zoom.imshow(
    cartesian_image,
    cmap="gray",
    extent=cartesian_extent_ft,
    origin="lower",
    interpolation="nearest",
)
ax_zoom.set_title(
    f"Warped cartesian zoom: {zoom_y_min_ft:.1f}-{zoom_y_max_ft:.1f} ft from origin"
)
ax_zoom.set_xlabel("x (ft)")
ax_zoom.set_ylabel("forward range (ft)")
ax_zoom.set_aspect("equal")
ax_zoom.set_xlim(zoom_x_min_ft, zoom_x_max_ft)
ax_zoom.set_ylim(zoom_y_min_ft, zoom_y_max_ft)
ax_zoom.set_xticks(grid_ticks_2ft(zoom_x_min_ft, zoom_x_max_ft))
ax_zoom.set_yticks(grid_ticks_2ft(zoom_y_min_ft, zoom_y_max_ft))
ax_zoom.grid(color="white", alpha=0.25, linewidth=0.8)
plt.show()

if used_polar_mask is not None:
    valid_polar_mask = np.zeros_like(polar_image, dtype=bool)
    valid_polar_mask[first_valid_range_row:last_valid_range_row + 1, :] = True
    polar_usage_view = np.full(polar_image.shape, np.nan, dtype=np.float32)
    polar_usage_view[valid_polar_mask & ~used_polar_mask] = 0
    polar_usage_view[valid_polar_mask & used_polar_mask] = 1

    fig_usage, ax_usage = plt.subplots(figsize=(10, 6), constrained_layout=True)
    usage_cmap = mcolors.ListedColormap(["red", "blue"])
    usage_cmap.set_bad(color=(0, 0, 0, 0))
    usage_norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5], usage_cmap.N)
    ax_usage.imshow(
        polar_image,
        cmap="gray",
        aspect="auto",
        extent=polar_extent,
        origin="upper",
        interpolation="nearest",
    )
    usage_image = ax_usage.imshow(
        polar_usage_view,
        cmap=usage_cmap,
        norm=usage_norm,
        alpha=0.45,
        aspect="auto",
        extent=polar_extent,
        origin="upper",
        interpolation="nearest",
    )
    ax_usage.set_title("Polar datapoints used by nearest-neighbor warp")
    ax_usage.set_xlabel("theta (degrees)")
    ax_usage.set_ylabel("range (ft)")
    if selected_range_ft is not None:
        ax_usage.axhline(selected_range_ft, color="magenta", linewidth=1.2, alpha=0.9)
    usage_colorbar = fig_usage.colorbar(usage_image, ax=ax_usage, ticks=[0, 1])
    usage_colorbar.ax.set_yticklabels(["unused", "used"])
    plt.show()
